In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import torch
import torch.nn as nn

In [2]:
# Preprocess new data
new_data_path = "C:/Users/Tharunraj_S/Desktop/Mini_project/Codes/test.csv"
df_new=pd.read_csv(new_data_path)
df_new.head()

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,4,51,80,1.01,2,0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [3]:
# dropping id column
df_new.drop('id', axis = 1, inplace = True)

In [4]:
df_new.drop('classification', axis = 1, inplace = True)

In [5]:
# rename column names to make it more user-friendly
df_new.columns = ['age', 'blood_pressure', 'specific_gravity', 'albumin', 'sugar', 'red_blood_cells', 'pus_cell',
              'pus_cell_clumps', 'bacteria', 'blood_glucose_random', 'blood_urea', 'serum_creatinine', 'sodium',
              'potassium', 'haemoglobin', 'packed_cell_volume', 'white_blood_cell_count', 'red_blood_cell_count',
              'hypertension', 'diabetes_mellitus', 'coronary_artery_disease', 'appetite', 'peda_edema',
              'anemia']

In [6]:
# replace some incorrect values
df_new['diabetes_mellitus'].replace(to_replace = {'\tno':'no','\tyes':'yes',' yes':'yes'},inplace=True)
df_new['coronary_artery_disease'] = df_new['coronary_artery_disease'].replace(to_replace = '\tno', value='no')

In [7]:
# converting necessary columns to numerical type
df_new['packed_cell_volume'] = pd.to_numeric(df_new['packed_cell_volume'], errors='coerce')
df_new['white_blood_cell_count'] = pd.to_numeric(df_new['white_blood_cell_count'], errors='coerce')
df_new['red_blood_cell_count'] = pd.to_numeric(df_new['red_blood_cell_count'], errors='coerce')

In [8]:
df_new


,age,blood_pressure,specific_gravity,albumin,sugar,red_blood_cells,pus_cell,pus_cell_clumps,bacteria,blood_glucose_random,...,haemoglobin,packed_cell_volume,white_blood_cell_count,red_blood_cell_count,hypertension,diabetes_mellitus,coronary_artery_disease,appetite,peda_edema,anemia
0,51,80,1.01,2,0,normal,normal,notpresent,notpresent,106,...,11.6,35,7300,4.6,no,no,no,good,no,no


In [9]:
column_mappings = {
    'red_blood_cells': {'normal': 0, 'abnormal': 1},
    'pus_cell': {'normal': 0, 'abnormal': 1},
    'pus_cell_clumps': {'notpresent': 0, 'present': 1},
    'bacteria': {'notpresent': 0, 'present': 1},
    'hypertension': {'no': 0, 'yes': 1},
    'diabetes_mellitus': {'no': 0, 'yes': 1},
    'coronary_artery_disease': {'no': 0, 'yes': 1},
    'appetite': {'good': 0, 'poor': 1},
    'peda_edema': {'no': 0, 'yes': 1},
    'anemia': {'no': 0, 'yes': 1}
}

# Apply custom mappings to the selected columns
df_new.replace(column_mappings, inplace=True)
'''
# Save the modified data to a new CSV file
df_new.to_csv('modified_data.csv', index=False)
'''

"\n# Save the modified data to a new CSV file\ndf_new.to_csv('modified_data.csv', index=False)\n"

In [10]:
df_new

,age,blood_pressure,specific_gravity,albumin,sugar,red_blood_cells,pus_cell,pus_cell_clumps,bacteria,blood_glucose_random,...,haemoglobin,packed_cell_volume,white_blood_cell_count,red_blood_cell_count,hypertension,diabetes_mellitus,coronary_artery_disease,appetite,peda_edema,anemia
0,51,80,1.01,2,0,0,0,0,0,106,...,11.6,35,7300,4.6,0,0,0,0,0,0


In [11]:
import numpy as np

#Data Normalization
# Convert Series to numpy array
biomarker_inputs = df_new.to_numpy()

y = ["ckd","notckd"]

# Define default values for each column
column_defaults = {
    'age': 51.483376,
    'blood_pressure': 76.469072,
    'specific_gravity': 1.017408,
    'albumin': 0,  # Example albumin level
    'sugar': 0,  # Example sugar level
    'red_blood_cells':0 ,  # Example red blood cell presence (1 for present, 0 for absent)
    'pus_cell': 1,  # Example pus cell presence (1 for present, 0 for absent)
    'pus_cell_clumps': 0,  # Example pus cell clumps presence (1 for present, 0 for absent)
    'bacteria': 0,  # Example bacteria presence (1 for present, 0 for absent)
    'blood_glucose_random': 120,
    'blood_urea': 57.425722,
    'serum_creatinine': 0.450142,
    'sodium': 137.528754,
    'potassium': 4.627244,
    'haemoglobin': 12.526437,
    'packed_cell_volume': 45,  # Example packed cell volume percentage
    'white_blood_cell_count': 7000,  # Example white blood cell count cells/μL
    'red_blood_cell_count': 5,  # Example red blood cell count million cells/μL
    'hypertension': 0,  # Example hypertension presence (1 for yes, 0 for no)
    'diabetes_mellitus': 0,  # Example diabetes mellitus presence (1 for yes, 0 for no)
    'coronary_artery_disease': 0,  # Example coronary artery disease presence (1 for yes, 0 for no)
    'appetite': 0,  # Example appetite (0 for good, 1 for poor)
    'peda_edema': 0,  # Example pedal edema presence (1 for yes, 0 for no)
    'anemia': 0  # Example anemia presence (1 for yes, 0 for no)

}

# Replace null values with custom default values using an if-else condition
for col_idx, col_name in enumerate(column_defaults.keys()):
    default_value = column_defaults[col_name]
    biomarker_inputs[:, col_idx] = np.where(np.isnan(biomarker_inputs[:, col_idx]),
                                             default_value, biomarker_inputs[:, col_idx])
    
# Convert Series to numpy array and class mapping for detection
y_test_array = np.array(y)
class_mapping = {'ckd': 0, 'notckd': 1}
y_test_indices = [class_mapping[label] for label in y_test_array]



In [12]:
input_size = 24 # number of biomarkers
hidden_size = 64 # choose an appropriate value
output_size=num_classes = 2

In [13]:
class KidneyCNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(KidneyCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)
        self.fc1 = nn.Linear(64 * (input_size - 4), 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # Add a channel dimension (batch_size, 1, input_size)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [14]:

# Convert to PyTorch tensors
X_test_tensor = torch.tensor(biomarker_inputs, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_indices, dtype=torch.long)


In [19]:

# Load the model architecture
model_path = 'C:/Users/Tharunraj_S/Desktop/Mini_project/Codes/Final/trained_model2.pt'
loaded_model = KidneyCNN(input_size,num_classes)
loaded_model.load_state_dict(torch.load(model_path))
loaded_model.eval()  # Set the model to evaluation mode

# Perform predictions
with torch.no_grad():
    outputs = loaded_model(X_test_tensor)
    
    
    _, predicted_classes = torch.max(outputs, 1)  # Get the predicted classes

# Convert tensor to Python list
predicted_classes_list = predicted_classes.tolist()
print('Predicted classes:', predicted_classes_list)


Predicted classes: [0]


In [20]:
outputs

tensor([[ 8.0174, -8.8469]])

In [21]:
class_probs = torch.softmax(outputs, dim=1)

predicted_class_index = torch.argmax(class_probs, dim=1).item()
confidence_score = class_probs[0, predicted_class_index].item()

# Get the predicted and actual class labels
predicted_class = list(class_mapping.keys())[list(class_mapping.values()).index(predicted_class_index)]

print(f"Predicted class: {predicted_class}")
print(f"Confidence score: {confidence_score:.4f}")

Predicted class: ckd
Confidence score: 1.0000


In [25]:

# Load the model architecture
model_path = 'C:/Users/Tharunraj_S/Desktop/Mini_project/Codes/Final/trained_model.pt'
loaded_model = KidneyCNN(input_size,num_classes)
loaded_model.load_state_dict(torch.load(model_path))
loaded_model.eval()  # Set the model to evaluation mode

# Perform predictions
with torch.no_grad():
    outputs = loaded_model(X_test_tensor)
    
    
    _, predicted_classes = torch.max(outputs, 1)  # Get the predicted classes

# Convert tensor to Python list
predicted_classes_list = predicted_classes.tolist()
print('Predicted classes:', predicted_classes_list)
class_probs = torch.softmax(outputs, dim=1)

predicted_class_index = torch.argmax(class_probs, dim=1).item()
confidence_score = class_probs[0, predicted_class_index].item()

# Get the predicted and actual class labels
predicted_class = list(class_mapping.keys())[list(class_mapping.values()).index(predicted_class_index)]

print(f"Predicted class: {predicted_class}")
print(f"Confidence score: {confidence_score:.4f}")

Predicted classes: [0]
Predicted class: ckd
Confidence score: 0.6508
